# Setup

In [159]:
import io
import requests
import rasterio
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.wkt import loads

from beak.experimental.utilities.io import check_path, data_folder
from beak.experimental.utilities.conversions import create_binary_raster


In [160]:
# Set base paths and files
BASE_PATH = data_folder()
EPSG_CODE = 102008
RESOLUTION = 500

BASE_EXTENT = "tusk_great_basin"
BASE_RASTER = BASE_PATH / "processed" / str(f"regional_{BASE_EXTENT}_{EPSG_CODE}_{RESOLUTION}") / "base_raster" / "template_raster.tif"
CMA_EXTENT = "regional"

OUT_FEATURES_PATH = BASE_PATH / "raw" / "mineral_deposits" / "Tungsten_Skarn" / "regional_tusk_great_basin" / "h4_tungsten_sites_0821a.gpkg"

OUT_FILE = BASE_PATH / "processed" / str(f"{CMA_EXTENT}_{BASE_EXTENT}_{EPSG_CODE}_{RESOLUTION}") / "labels" / "TA2_cdr_0821a.tif"

base_raster = rasterio.open(BASE_RASTER)
check_path(OUT_FILE.parent)

print(f"Export: {OUT_FILE}")


Export: S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\data\processed\regional_tusk_great_basin_102008_500\labels\TA2_cdr_0821a.tif


In [161]:
token = ""


# Pull data from CDR

In [162]:
url = 'https://api.cdr.land/v1/minerals/dedup-site/search/Tungsten?with_location=true&with_deposit_types_only=true&system=minmod&top_n=1&limit=-1'

headers = {
    'accept': 'application/json',
    'Authorization': f'Bearer {token}'
}

response = requests.get(url, headers=headers)
data = response.text


In [163]:
file = io.StringIO(data)
df = pd.read_csv(file)
df


,id,record_ids,mineral_site_ids,names,type,rank,country,province,crs,centroid_epsg_4326,...,contained_metal_unit,tonnage,tonnage_unit,grade,grade_unit,top1_deposit_type,top1_deposit_group,top1_deposit_environment,top1_deposit_classification_confidence,top1_deposit_classification_source
0,0008fca217e440bdb0986c035c6e271d,"[""3b0bd9b05a53488ab06d7247c7bf258d"", ""94093334...","[""https://minmod.isi.edu/resource/site__mrdata...",San Jorge,Occurrence,"[""C"", ""E""]",Mexico,Baja California,EPSG:4326,POINT (-116.03413499999999 32.133404999999996),...,million tonnes,NaN,million tonnes,NaN,percent,Skarn tungsten ± Mo,Skarn,Magmatic hydrothermal,1.000000,"algorithm predictions, SRI crosswalk agent v0"
1,0013acb7bec7409d8e105a98d621751b,099637f11d9b40c293f2402a2ed6aff8,https://minmod.isi.edu/resource/site__mrdata-u...,Jones Ranch,Unknown,D,United States,California,EPSG:4326,POINT (-119.73427 37.32353),...,million tonnes,NaN,million tonnes,NaN,percent,Skarn tungsten ± Mo,Skarn,Magmatic hydrothermal,1.000000,"algorithm predictions, SRI crosswalk agent v0"
2,0024d6bbb11a4fd3b37ea164cb4e9586,"[""d911d05430984873b8aa5472b9a72cad"", ""b7bd0e7c...",https://minmod.isi.edu/resource/site__mrdata-u...,"[""Pamlico"", ""Silver Star Tungsten-Copper""]",Past Producer,D,United States,Nevada,EPSG:4326,POINT (-118.49036 38.48102),...,million tonnes,NaN,million tonnes,NaN,percent,Greisen tungsten- molybdenum±Bi,Greisen,Magmatic hydrothermal,0.348706,"algorithm predictions, SRI deposit type classi..."
3,0031ac6bb6ba44de935d71fa903d05f9,1e7c3493fa70455b95bd6ea3baf587c7,https://minmod.isi.edu/resource/site__mrdata-u...,King Midas,Unknown,D,United States,Nevada,EPSG:4326,POINT (-114.96777 39.92413),...,million tonnes,NaN,million tonnes,NaN,percent,Greisen tungsten- molybdenum±Bi,Greisen,Magmatic hydrothermal,0.656400,"algorithm predictions, SRI deposit type classi..."
4,00336ce0449f402dbebe75f474e587e5,6305f3ae56ec409db7a675297b3e9947,https://minmod.isi.edu/resource/site__mrdata-u...,B. Estelle Mine,Prospect,B,United States,Utah,EPSG:4326,POINT (-113.85307 40.19049),...,million tonnes,NaN,million tonnes,NaN,percent,Skarn tungsten ± Mo,Skarn,Magmatic hydrothermal,0.401198,"algorithm predictions, SRI deposit type classi..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6371,ffc620d24a014420b3168b228af0038f,"[""acccc313f12340d5ac85ab305f4b118c"", ""6a5ec30a...","[""https://minmod.isi.edu/resource/site__mrdata...",Thorne Tungsten Property,Past Producer,"[""E"", ""D""]",United States,Nevada,EPSG:4326,POINT (-118.31332499999999 39.009055000000004),...,million tonnes,NaN,million tonnes,NaN,percent,Skarn tungsten ± Mo,Skarn,Magmatic hydrothermal,1.000000,"algorithm predictions, SRI crosswalk agent v0"
6372,ffc77d31aa364262bd161c5196b942d6,b924aafcb6084282840ae985a0bd8fae,https://minmod.isi.edu/resource/site__mrdata-u...,Bayan,Producer,D,Kazakhstan,NaN,EPSG:4326,POINT (68.00099 51.99788),...,million tonnes,NaN,million tonnes,NaN,percent,Greisen tungsten- molybdenum±Bi,Greisen,Magmatic hydrothermal,0.642518,"algorithm predictions, SRI deposit type classi..."
6373,ffd4d2f18ef74b6489744324bdcf3a15,c633239f92a643bc9721c1f3a253fd99,https://minmod.isi.edu/resource/site__mrdata-u...,Eldorado Creek,Past Producer,B,United States,Alaska,EPSG:4326,POINT (-163.76765 64.58402),...,million tonnes,NaN,million tonnes,NaN,percent,Shoreline placer gold,Placer,Erosional,0.292707,"algorithm predictions, SRI deposit type classi..."
6374,fffa1a22ebfd402693c63e6a0896f88a,77b27ad70e3e49bfa4689cdf547e510a,https://minmod.isi.edu/resource/site__doi-org-...,Homestake Creek,Occurrence,U,United States,Alaska,EPSG:4267,POINT (164.82 65.684),...,million tonnes,NaN,million tonnes,NaN,percent,Fluvial placer gold,Placer,Erosional,0.939675,"algorithm predictions, SRI deposit type classi..."


## Create geodataframe and clear data

In [164]:
mineral_sites = df.copy()
mineral_sites["geometry"] = mineral_sites["centroid_epsg_4326"].apply(loads)
mineral_sites = gpd.GeoDataFrame(mineral_sites, crs="EPSG:4326")
mineral_sites = mineral_sites.explode(column="geometry", ignore_index=True)
mineral_sites = mineral_sites.drop_duplicates(subset=["geometry"])
mineral_sites


,id,record_ids,mineral_site_ids,names,type,rank,country,province,crs,centroid_epsg_4326,...,tonnage,tonnage_unit,grade,grade_unit,top1_deposit_type,top1_deposit_group,top1_deposit_environment,top1_deposit_classification_confidence,top1_deposit_classification_source,geometry
0,0008fca217e440bdb0986c035c6e271d,"[""3b0bd9b05a53488ab06d7247c7bf258d"", ""94093334...","[""https://minmod.isi.edu/resource/site__mrdata...",San Jorge,Occurrence,"[""C"", ""E""]",Mexico,Baja California,EPSG:4326,POINT (-116.03413499999999 32.133404999999996),...,NaN,million tonnes,NaN,percent,Skarn tungsten ± Mo,Skarn,Magmatic hydrothermal,1.000000,"algorithm predictions, SRI crosswalk agent v0",POINT (-116.03413 32.13340)
1,0013acb7bec7409d8e105a98d621751b,099637f11d9b40c293f2402a2ed6aff8,https://minmod.isi.edu/resource/site__mrdata-u...,Jones Ranch,Unknown,D,United States,California,EPSG:4326,POINT (-119.73427 37.32353),...,NaN,million tonnes,NaN,percent,Skarn tungsten ± Mo,Skarn,Magmatic hydrothermal,1.000000,"algorithm predictions, SRI crosswalk agent v0",POINT (-119.73427 37.32353)
2,0024d6bbb11a4fd3b37ea164cb4e9586,"[""d911d05430984873b8aa5472b9a72cad"", ""b7bd0e7c...",https://minmod.isi.edu/resource/site__mrdata-u...,"[""Pamlico"", ""Silver Star Tungsten-Copper""]",Past Producer,D,United States,Nevada,EPSG:4326,POINT (-118.49036 38.48102),...,NaN,million tonnes,NaN,percent,Greisen tungsten- molybdenum±Bi,Greisen,Magmatic hydrothermal,0.348706,"algorithm predictions, SRI deposit type classi...",POINT (-118.49036 38.48102)
3,0031ac6bb6ba44de935d71fa903d05f9,1e7c3493fa70455b95bd6ea3baf587c7,https://minmod.isi.edu/resource/site__mrdata-u...,King Midas,Unknown,D,United States,Nevada,EPSG:4326,POINT (-114.96777 39.92413),...,NaN,million tonnes,NaN,percent,Greisen tungsten- molybdenum±Bi,Greisen,Magmatic hydrothermal,0.656400,"algorithm predictions, SRI deposit type classi...",POINT (-114.96777 39.92413)
4,00336ce0449f402dbebe75f474e587e5,6305f3ae56ec409db7a675297b3e9947,https://minmod.isi.edu/resource/site__mrdata-u...,B. Estelle Mine,Prospect,B,United States,Utah,EPSG:4326,POINT (-113.85307 40.19049),...,NaN,million tonnes,NaN,percent,Skarn tungsten ± Mo,Skarn,Magmatic hydrothermal,0.401198,"algorithm predictions, SRI deposit type classi...",POINT (-113.85307 40.19049)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6371,ffc620d24a014420b3168b228af0038f,"[""acccc313f12340d5ac85ab305f4b118c"", ""6a5ec30a...","[""https://minmod.isi.edu/resource/site__mrdata...",Thorne Tungsten Property,Past Producer,"[""E"", ""D""]",United States,Nevada,EPSG:4326,POINT (-118.31332499999999 39.009055000000004),...,NaN,million tonnes,NaN,percent,Skarn tungsten ± Mo,Skarn,Magmatic hydrothermal,1.000000,"algorithm predictions, SRI crosswalk agent v0",POINT (-118.31332 39.00906)
6372,ffc77d31aa364262bd161c5196b942d6,b924aafcb6084282840ae985a0bd8fae,https://minmod.isi.edu/resource/site__mrdata-u...,Bayan,Producer,D,Kazakhstan,NaN,EPSG:4326,POINT (68.00099 51.99788),...,NaN,million tonnes,NaN,percent,Greisen tungsten- molybdenum±Bi,Greisen,Magmatic hydrothermal,0.642518,"algorithm predictions, SRI deposit type classi...",POINT (68.00099 51.99788)
6373,ffd4d2f18ef74b6489744324bdcf3a15,c633239f92a643bc9721c1f3a253fd99,https://minmod.isi.edu/resource/site__mrdata-u...,Eldorado Creek,Past Producer,B,United States,Alaska,EPSG:4326,POINT (-163.76765 64.58402),...,NaN,million tonnes,NaN,percent,Shoreline placer gold,Placer,Erosional,0.292707,"algorithm predictions, SRI deposit type classi...",POINT (-163.76765 64.58402)
6374,fffa1a22ebfd402693c63e6a0896f88a,77b27ad70e3e49bfa4689cdf547e510a,https://minmod.isi.edu/resource/site__doi-org-...,Homestake Creek,Occurrence,U,United States,Alaska,EPSG:4267,POINT (164.82 65.684),...,NaN,million tonnes,NaN,percent,Fluvial placer gold,Placer,Erosional,0.939675,"algorithm predictions, SRI deposit type classi...",POINT (164.82000 65.68400)


Export

In [165]:
out_file_csv = OUT_FEATURES_PATH.parent / "mineral_sites_01_no_filter_0821a.csv"
out_file_gpkg = OUT_FEATURES_PATH 

mineral_sites.to_csv(out_file_csv, index=False)
mineral_sites.to_file(out_file_gpkg, driver="GPKG", layer="mineral_sites_01_no_filter_0821_a_centroid")


# Filtering

## Basic filter

In [166]:
QUERY = (
    "top1_deposit_classification_confidence >= 0.5 and "
    "type.str.contains('Producer|Prospect|NotSpecified', regex=True) and "
    "rank.str.contains('A|B|C|U', regex=True) and "
    "country == 'United States'"
)

mineral_sites_filtered = mineral_sites.copy()
mineral_sites_filtered = mineral_sites_filtered.query(QUERY)
mineral_sites_filtered


,id,record_ids,mineral_site_ids,names,type,rank,country,province,crs,centroid_epsg_4326,...,tonnage,tonnage_unit,grade,grade_unit,top1_deposit_type,top1_deposit_group,top1_deposit_environment,top1_deposit_classification_confidence,top1_deposit_classification_source,geometry
13,0061c01e19574715a72a435e67b031c6,"[""4d005eee951745a6a87d63bbe1ce3be2"", ""80fdd843...","[""https://minmod.isi.edu/resource/site__mrdata...","[""Midway Prospect"", ""Midway""]","[""Prospect"", ""Past Producer""]","[""C"", ""D""]",United States,Arizona,EPSG:4326,POINT (-113.82607 34.938645),...,NaN,million tonnes,NaN,percent,Greisen tungsten- molybdenum±Bi,Greisen,Magmatic hydrothermal,0.879643,"algorithm predictions, SRI deposit type classi...",POINT (-113.82607 34.93865)
24,01112777118b4d959f9ecd38292f1a5d,566537f959ce4da7863a537ce4aa5c24,https://minmod.isi.edu/resource/site__mrdata-u...,Robinette Prospect,Past Producer,B,United States,Nevada,EPSG:4326,POINT (-115.08423 41.90823),...,NaN,million tonnes,NaN,percent,Skarn tungsten ± Mo,Skarn,Magmatic hydrothermal,1.000000,"algorithm predictions, SRI crosswalk agent v0",POINT (-115.08423 41.90823)
32,0151e1af47e447eeaab99dd00f3836a3,"[""b0c395c3c5b1418182d9081d904f0589"", ""526a5a8b...","[""https://minmod.isi.edu/resource/site__mrdata...",Wolfram Lode,"[""Past Producer"", ""Occurrence""]","[""C"", ""D""]",United States,South Dakota,EPSG:4326,POINT (-103.59221500000001 43.937195),...,NaN,million tonnes,NaN,percent,Greisen tungsten- molybdenum±Bi,Greisen,Magmatic hydrothermal,0.821199,"algorithm predictions, SRI deposit type classi...",POINT (-103.59222 43.93720)
36,0169d9f898a5413cadb2233b5288a9d2,9c28ca900c7b4fb7ba590c004f82fd18,https://minmod.isi.edu/resource/site__mrdata-u...,Big Hurrah Creek Placer,Past Producer,B,United States,Alaska,EPSG:4326,POINT (-164.26994 64.65209),...,NaN,million tonnes,NaN,percent,Fluvial placer gold,Placer,Erosional,0.563810,"algorithm predictions, SRI deposit type classi...",POINT (-164.26994 64.65209)
39,0175c392bec742e0b35958ad0f13c0d8,"[""5d67d56f7fc44e309d5f1bd21648feaf"", ""78f50dc5...","[""https://minmod.isi.edu/resource/site__mrdata...",Craigie Creek,"[""Prospect"", ""Occurrence""]",C,United States,Alaska,EPSG:4326,POINT (-149.38924 61.78345),...,NaN,million tonnes,NaN,percent,Greisen tungsten- molybdenum±Bi,Greisen,Magmatic hydrothermal,0.617716,"algorithm predictions, SRI deposit type classi...",POINT (-149.38924 61.78345)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6340,fedc6450c3e447448eefc34f5b9072dc,"[""9dfd8bb8c4284820928a0a42305cb1be"", ""73eb761c...","[""https://minmod.isi.edu/resource/site__mrdata...","[""Big Falls Creek Deposit"", ""Little Falls Cree...",Past Producer,B,United States,Idaho,EPSG:4326,POINT (-114.243255 43.855795),...,NaN,million tonnes,NaN,percent,Greisen tungsten- molybdenum±Bi,Greisen,Magmatic hydrothermal,0.726251,"algorithm predictions, SRI deposit type classi...",POINT (-114.24326 43.85580)
6348,ff2270cf33d64879b4bc725e864f06db,"[""c29b10aa07a0437085c53f53de30f4d1"", ""ef475710...","[""https://minmod.isi.edu/resource/site__mrdata...",Nome Creek Area,Past Producer,B,United States,Alaska,EPSG:4326,POINT (-146.70283 65.34167),...,NaN,million tonnes,NaN,percent,Fluvial placer gold,Placer,Erosional,1.000000,"algorithm predictions, SRI crosswalk agent v0",POINT (-146.70283 65.34167)
6355,ff64ef9b7c684db3b885df780d044f0e,201a5f4b4b0a4dc38bba77a2ecb767c2,https://minmod.isi.edu/resource/site__mrdata-u...,Hope Creek,Prospect,B,United States,Alaska,EPSG:4326,POINT (-146.35696 65.39971),...,NaN,million tonnes,NaN,percent,Fluvial placer gold,Placer,Erosional,1.000000,"algorithm predictions, SRI crosswalk agent v0",POINT (-146.35696 65.39971)
6363,ff881c04f70f4f8099f78895577326e5,"[""01ac7963ee8f4b9a91cb7f605ecb16a2"", ""71860248...","[""https://minmod.isi.edu/resource/site__mrdata...",Rare Metals,Past Producer,"[""B"", ""D""]",United States,Nevada,EPSG:4326,POINT (-114.29906 40.91285),...,NaN,million tonnes

Export

In [167]:
out_file_csv = OUT_FEATURES_PATH.parent / "mineral_sites_02_basic_filter_0821a.csv"
out_file_gpkg = OUT_FEATURES_PATH

mineral_sites_filtered.to_csv(out_file_csv, index=False)
mineral_sites_filtered.to_file(out_file_gpkg, driver="GPKG", layer="mineral_sites_02_basic_filter_0821a_centroid")

In [168]:
column = "top1_deposit_group"
deposit_types = mineral_sites_filtered[column].unique()
deposit_types = pd.DataFrame(deposit_types, columns=[column])
deposit_types

,top1_deposit_group
0,Greisen
1,Skarn
2,Placer
3,Vein
4,Replacement
5,Orogenic
6,Metamorphic
7,Supergene
8,Porphyry
9,Epithermal


## Final filtering

In [169]:
QUERY = (
    "top1_deposit_group >= 'Skarn' and "
    "top1_deposit_classification_confidence >= 0.5 and "
    "type.str.contains('Producer|Prospect|NotSpecified', regex=True) and "
    "rank.str.contains('A|B|C|U', regex=True) and "
    "country == 'United States'"
)

mineral_sites_filtered = mineral_sites.copy()
mineral_sites_filtered = mineral_sites_filtered.query(QUERY)
mineral_sites_filtered

,id,record_ids,mineral_site_ids,names,type,rank,country,province,crs,centroid_epsg_4326,...,tonnage,tonnage_unit,grade,grade_unit,top1_deposit_type,top1_deposit_group,top1_deposit_environment,top1_deposit_classification_confidence,top1_deposit_classification_source,geometry
24,01112777118b4d959f9ecd38292f1a5d,566537f959ce4da7863a537ce4aa5c24,https://minmod.isi.edu/resource/site__mrdata-u...,Robinette Prospect,Past Producer,B,United States,Nevada,EPSG:4326,POINT (-115.08423 41.90823),...,NaN,million tonnes,NaN,percent,Skarn tungsten ± Mo,Skarn,Magmatic hydrothermal,1.000000,"algorithm predictions, SRI crosswalk agent v0",POINT (-115.08423 41.90823)
46,01ab7512d96948a88501e8b496490387,"[""087b66db713b4bb09a11259772cb877f"", ""a7f63a4f...","[""https://minmod.isi.edu/resource/site__mrdata...","[""Eagle Peak"", ""Eagle Peak Mine""]","[""Past Producer"", ""Occurrence""]",B,United States,Washington,EPSG:4326,POINT (-121.78119 46.75564),...,NaN,million tonnes,NaN,percent,Vein polymetallic,Vein,Magmatic hydrothermal,1.000000,"algorithm predictions, SRI crosswalk agent v0",POINT (-121.78119 46.75564)
51,01eaea0ba47f4f6e88ab6d9d52abe42b,"[""4d7ab60eb2644d5396064d163fe0bf6b"", ""2f8b3790...","[""https://minmod.isi.edu/resource/site__mrdata...","[""Bright Star Prospect"", ""Bright Star""]","[""Prospect"", ""Occurrence""]","[""C"", ""D""]",United States,California,EPSG:4326,POINT (-118.34259 35.61694),...,NaN,million tonnes,NaN,percent,Vein tungsten,Vein,Magmatic hydrothermal,1.000000,"algorithm predictions, SRI crosswalk agent v0",POINT (-118.34259 35.61694)
54,020c32b5082e4d368539df12e57426de,658de568c1d241e1aee4b1356716dc48,https://minmod.isi.edu/resource/site__mrdata-u...,Juno -Echo Mine,Past Producer,B,United States,Washington,EPSG:4326,POINT (-117.68471 48.29096),...,NaN,million tonnes,NaN,percent,Skarn molybdenum,Skarn,Magmatic hydrothermal,0.757692,"algorithm predictions, SRI deposit type classi...",POINT (-117.68471 48.29096)
66,027d3b1a7176486184a6eec639bc17d2,07ac4248389f45f885b05db9bc54b0a3,https://minmod.isi.edu/resource/site__doi-org-...,Baldy,Prospect,U,United States,Alaska,EPSG:4267,POINT (135.0361 57.7961),...,NaN,million tonnes,NaN,percent,Skarn copper,Skarn,Magmatic hydrothermal,0.681801,"algorithm predictions, SRI deposit type classi...",POINT (135.03610 57.79610)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6323,fdee1127b5c844d4a2433520e770a300,"[""06f992323eda4aa7b14f9c07911ec832"", ""92207e49...","[""https://minmod.isi.edu/resource/site__mrdata...",Riverside,"[""Unknown"", ""Past Producer""]","[""B"", ""C""]",United States,Alaska,EPSG:4326,POINT (-130.06438 56.0017),...,NaN,million tonnes,NaN,percent,Vein polymetallic,Vein,Magmatic hydrothermal,1.000000,"algorithm predictions, SRI crosswalk agent v0",POINT (-130.06438 56.00170)
6333,feb60f7c04624a02a8b759c0129a28ec,"[""6d2635d9a8e44d1ebf511f2caaf910e2"", ""f2e05a90...","[""https://minmod.isi.edu/resource/site__mrdata...",Snod Grass,Past Producer,C,United States,California,EPSG:4326,POINT (-119.72377 38.66491),...,NaN,million tonnes,NaN,percent,Skarn tungsten ± Mo,Skarn,Magmatic hydrothermal,1.000000,"algorithm predictions, SRI crosswalk agent v0",POINT (-119.72377 38.66491)
6337,fed4ff66d030433fa9cd9738b13ee091,"[""dd9a01e9c51c418f90caa97842c5a5d7"", ""fffc79a3...","[""https://minmod.isi.edu/resource/site__mrdata...","[""Alex Eske Tungsten"", ""Alex Eske Mine""]","[""Past Producer"", ""Occurrence""]",C,United States,Nevada,EPSG:4326,POINT (-119.64752999999999 39.10893),...,NaN,million tonnes,NaN,percent,Skarn tungsten ± Mo,Skarn,Magmatic hydrothermal,1.000000,"algorithm predictions, SRI crosswalk agent v0",POINT (-119.64753 39.10893)
6363,ff881c04f70f4f8099f78895577326e5,"[""01ac7963ee8f4b9a91cb7f605ecb16a2"", ""71860248...","[""https://minmod.isi.edu/resource/site__mrdata...",Rare Metals,Past Producer,"[""B"", ""D""]",United States,Nevada,EPSG:4326,POINT (-114.29906 40.91285),...,NaN,million tonnes,NaN,percent,Vein tungsten,

Export

In [170]:
out_file_csv =  OUT_FEATURES_PATH.parent / "mineral_sites_03_final_filter_0821a.csv"
out_file_gpkg = OUT_FEATURES_PATH

mineral_sites_filtered.to_csv(out_file_csv, index=False)
mineral_sites_filtered.to_file(out_file_gpkg, driver="GPKG", layer="mineral_sites_03_final_filter_0821a_centroid")

# Create raster

In [171]:
data = mineral_sites_filtered.copy()
data = data.explode(ignore_index=True)
data = data[~data.is_empty]
data = data.reset_index(drop=True)
data = data.to_crs(base_raster.crs)


In [172]:
labels_array = create_binary_raster(data, base_raster, all_touched=False, same_shape=True, fill_negatives=True, out_file=OUT_FILE)
print(f"Number of positive training labels: {np.sum(labels_array==1)}")

File already exists: TA2_cdr_0821a.tif.
Number of positive training labels: 438
